In [2]:
# =====================================================================
# MONTHLY PREDICTOR STACK CREATION
# Flexible filename detection + alignment verification + logging
#
# Supported filename examples:
#   2017_01.tif
#   2017_1.tif
#   LST_Day_2017_1.tif
#   NDVI_2017_01.tif
#   CHIRPS_2017-01.tif
#
# Expected project structure:
#
# data/
# └── processed/
#     └── rasters_aligned/
#         ├── monthly/
#         │   ├── CCS/
#         │   ├── CDR/
#         │   ├── CHIRPS/
#         │   ├── ERA5/
#         │   ├── GSMaP_Gauge/
#         │   ├── GSMaP_MVK/
#         │   ├── IMERG/
#         │   ├── LST/
#         │   ├── NDVI/
#         │   ├── PDIR/
#         │   └── PERSIANN/
#         └── static/
#
# Output:
# data/processed/predictor_stack/
# =====================================================================


# =====================================================================
# 1. IMPORT LIBRARIES
# =====================================================================

from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import Affine


warnings.filterwarnings("ignore")


# =====================================================================
# 2. PROJECT CONFIGURATION
# =====================================================================

PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling"
    r"\Precipitation-Downscaling-Khulna"
)

ALIGNED_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rasters_aligned"
)

MONTHLY_FOLDER = ALIGNED_FOLDER / "monthly"
STATIC_FOLDER = ALIGNED_FOLDER / "static"

OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "predictor_stack"
)

LOG_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "logs"
)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Study period
# ---------------------------------------------------------------------

START_YEAR = 2017
END_YEAR = 2022


# ---------------------------------------------------------------------
# Monthly datasets
#
# Folder names must match the folders inside:
# rasters_aligned/monthly/
# ---------------------------------------------------------------------

MONTHLY_DATASETS = [
    "CCS",
    "CDR",
    "CHIRPS",
    "ERA5",
    "GSMaP_Gauge",
    "GSMaP_MVK",
    "IMERG",
    "LST",
    "NDVI",
    "PDIR",
    "PERSIANN",
]


# ---------------------------------------------------------------------
# Optional datasets
#
# True:
# Dataset missing হলে warning দেবে, কিন্তু stack creation বন্ধ করবে না।
#
# False:
# Dataset missing হলে সেই মাস failed হবে।
#
# সাধারণত CDR বা PDIR-এ সমস্যা থাকলে সাময়িকভাবে optional করা যায়।
# ---------------------------------------------------------------------

OPTIONAL_MONTHLY_DATASETS = set([
    # "CDR",
    # "PDIR",
])


# ---------------------------------------------------------------------
# Static raster configuration
#
# AUTO_DISCOVER_STATIC = True হলে static folder-এর সব tif ব্যবহার করবে।
# ---------------------------------------------------------------------

AUTO_DISCOVER_STATIC = True


# ---------------------------------------------------------------------
# Output settings
# ---------------------------------------------------------------------

OUTPUT_DTYPE = "float32"
OUTPUT_NODATA = -9999.0
COMPRESSION = "deflate"

OVERWRITE_EXISTING = True
VERIFY_ALIGNMENT = True
STRICT_BOUNDS_CHECK = True

TRANSFORM_TOLERANCE = 1e-10
BOUNDS_TOLERANCE = 1e-8


# =====================================================================
# 3. HELPER FUNCTIONS
# =====================================================================

def normalize_text(text):
    """
    Normalize text for case-insensitive comparison.
    """
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def find_dataset_folder(parent_folder, dataset_name):
    """
    Find a dataset folder case-insensitively.

    Example:
        Requested: GSMaP_Gauge
        Existing:  gsmap gauge
    """

    parent_folder = Path(parent_folder)

    direct_path = parent_folder / dataset_name

    if direct_path.exists() and direct_path.is_dir():
        return direct_path

    target_normalized = normalize_text(dataset_name)

    for folder in parent_folder.iterdir():

        if not folder.is_dir():
            continue

        if normalize_text(folder.name) == target_normalized:
            return folder

    raise FileNotFoundError(
        f"Dataset folder পাওয়া যায়নি:\n"
        f"Dataset: {dataset_name}\n"
        f"Parent folder: {parent_folder}"
    )


def extract_year_month(filename):
    """
    Extract year and month from flexible filenames.

    Supported examples:
        2017_01.tif
        2017_1.tif
        LST_Day_2017_1.tif
        LST_Day_2017_01.tif
        NDVI-2017-01.tif
        CHIRPS_2017-1_monthly.tif

    Returns:
        (year, month) or None
    """

    filename = Path(filename).stem

    patterns = [
        # 2017_01, 2017-01, 2017.01
        r"(?<!\d)(20\d{2})[_\-.](0?[1-9]|1[0-2])(?!\d)",

        # 201701
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
    ]

    for pattern in patterns:

        match = re.search(pattern, filename)

        if match:

            year = int(match.group(1))
            month = int(match.group(2))

            if 1 <= month <= 12:
                return year, month

    return None


def build_monthly_file_index(dataset_folder):
    """
    Build an index:

        {
            (2017, 1): Path(...),
            (2017, 2): Path(...),
            ...
        }
    """

    dataset_folder = Path(dataset_folder)

    raster_files = sorted([
        path
        for path in dataset_folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in [".tif", ".tiff"]
    ])

    file_index = {}
    skipped_files = []
    duplicate_files = []

    for raster_path in raster_files:

        year_month = extract_year_month(raster_path.name)

        if year_month is None:
            skipped_files.append(raster_path)
            continue

        year, month = year_month
        key = (year, month)

        if key in file_index:

            duplicate_files.append({
                "Year": year,
                "Month": month,
                "First_File": str(file_index[key]),
                "Duplicate_File": str(raster_path),
            })

            # প্রথম file রাখবে
            continue

        file_index[key] = raster_path

    return file_index, skipped_files, duplicate_files


def sanitize_band_name(name):
    """
    Create a safe and readable raster band name.
    """

    name = str(name).strip()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^A-Za-z0-9_\-]+", "", name)

    if not name:
        name = "Unnamed_Band"

    return name[:100]


def discover_static_rasters(static_folder):
    """
    Discover all static tif files recursively.
    """

    static_folder = Path(static_folder)

    if not static_folder.exists():
        print(f"WARNING: Static folder পাওয়া যায়নি: {static_folder}")
        return []

    rasters = sorted([
        path
        for path in static_folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in [".tif", ".tiff"]
    ])

    return rasters


def affine_equal(transform_1, transform_2, tolerance=1e-10):
    """
    Compare two affine transforms with tolerance.
    """

    values_1 = np.array(tuple(transform_1), dtype=float)
    values_2 = np.array(tuple(transform_2), dtype=float)

    return np.allclose(
        values_1,
        values_2,
        atol=tolerance,
        rtol=0
    )


def bounds_equal(bounds_1, bounds_2, tolerance=1e-8):
    """
    Compare raster bounds with tolerance.
    """

    values_1 = np.array(tuple(bounds_1), dtype=float)
    values_2 = np.array(tuple(bounds_2), dtype=float)

    return np.allclose(
        values_1,
        values_2,
        atol=tolerance,
        rtol=0
    )


def get_raster_metadata(raster_path):
    """
    Read essential metadata from a raster.
    """

    with rasterio.open(raster_path) as src:

        return {
            "path": Path(raster_path),
            "crs": src.crs,
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "transform": src.transform,
            "bounds": src.bounds,
            "resolution": src.res,
            "nodata": src.nodata,
            "dtype": src.dtypes[0],
        }


def verify_raster_against_reference(
    raster_path,
    reference_metadata,
    check_bounds=True
):
    """
    Verify whether a raster has the same grid as the reference raster.

    Returns:
        list of alignment errors
    """

    errors = []

    with rasterio.open(raster_path) as src:

        if src.crs != reference_metadata["crs"]:
            errors.append(
                f"CRS mismatch: {src.crs} != "
                f"{reference_metadata['crs']}"
            )

        if src.width != reference_metadata["width"]:
            errors.append(
                f"Width mismatch: {src.width} != "
                f"{reference_metadata['width']}"
            )

        if src.height != reference_metadata["height"]:
            errors.append(
                f"Height mismatch: {src.height} != "
                f"{reference_metadata['height']}"
            )

        if not affine_equal(
            src.transform,
            reference_metadata["transform"],
            TRANSFORM_TOLERANCE
        ):
            errors.append(
                f"Transform mismatch: {src.transform} != "
                f"{reference_metadata['transform']}"
            )

        if check_bounds and not bounds_equal(
            src.bounds,
            reference_metadata["bounds"],
            BOUNDS_TOLERANCE
        ):
            errors.append(
                f"Bounds mismatch: {src.bounds} != "
                f"{reference_metadata['bounds']}"
            )

    return errors


def read_raster_as_float32(raster_path):
    """
    Read first band as float32.

    Source nodata, masked values, NaN and infinity are converted
    to the common output nodata value.
    """

    with rasterio.open(raster_path) as src:

        data = src.read(1, masked=True).astype(np.float32)

        # Masked values -> NaN
        data = data.filled(np.nan)

        # Infinity -> NaN
        data[~np.isfinite(data)] = np.nan

        # NaN -> common nodata
        data = np.where(
            np.isnan(data),
            OUTPUT_NODATA,
            data
        ).astype(np.float32)

    return data


def make_unique_band_names(names):
    """
    Ensure that all band names are unique.
    """

    output_names = []
    counters = {}

    for name in names:

        clean_name = sanitize_band_name(name)

        if clean_name not in counters:
            counters[clean_name] = 1
            output_names.append(clean_name)

        else:
            counters[clean_name] += 1

            unique_name = (
                f"{clean_name}_{counters[clean_name]}"
            )

            output_names.append(unique_name)

    return output_names


def get_month_list(start_year, end_year):
    """
    Create a list of all months between start and end year.
    """

    months = []

    for year in range(start_year, end_year + 1):

        for month in range(1, 13):

            months.append((year, month))

    return months


# =====================================================================
# 4. INITIAL VALIDATION
# =====================================================================

print("=" * 78)
print("MONTHLY PREDICTOR STACK CREATION")
print("=" * 78)

print(f"Project root   : {PROJECT_ROOT}")
print(f"Aligned folder : {ALIGNED_FOLDER}")
print(f"Monthly folder : {MONTHLY_FOLDER}")
print(f"Static folder  : {STATIC_FOLDER}")
print(f"Output folder  : {OUTPUT_FOLDER}")
print(f"Log folder     : {LOG_FOLDER}")
print()


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root পাওয়া যায়নি:\n{PROJECT_ROOT}"
    )

if not ALIGNED_FOLDER.exists():
    raise FileNotFoundError(
        f"Aligned folder পাওয়া যায়নি:\n{ALIGNED_FOLDER}"
    )

if not MONTHLY_FOLDER.exists():
    raise FileNotFoundError(
        f"Monthly folder পাওয়া যায়নি:\n{MONTHLY_FOLDER}"
    )


# =====================================================================
# 5. INDEX ALL MONTHLY RASTERS
# =====================================================================

monthly_indexes = {}
monthly_folder_paths = {}

dataset_inventory_records = []
skipped_file_records = []
duplicate_file_records = []


print("=" * 78)
print("INDEXING MONTHLY RASTERS")
print("=" * 78)


for dataset in MONTHLY_DATASETS:

    try:

        dataset_folder = find_dataset_folder(
            MONTHLY_FOLDER,
            dataset
        )

        monthly_folder_paths[dataset] = dataset_folder

        file_index, skipped_files, duplicate_files = (
            build_monthly_file_index(dataset_folder)
        )

        monthly_indexes[dataset] = file_index

        dataset_inventory_records.append({
            "Dataset": dataset,
            "Folder": str(dataset_folder),
            "Raster_Files_Indexed": len(file_index),
            "Skipped_Files": len(skipped_files),
            "Duplicate_Months": len(duplicate_files),
            "Status": "Found",
        })

        for skipped_path in skipped_files:

            skipped_file_records.append({
                "Dataset": dataset,
                "File": str(skipped_path),
                "Reason": "Year and month could not be extracted",
            })

        for duplicate_record in duplicate_files:

            duplicate_record["Dataset"] = dataset
            duplicate_file_records.append(duplicate_record)

        print(
            f"{dataset:<18}: "
            f"{len(file_index):>3} monthly raster(s)"
        )

    except Exception as error:

        monthly_indexes[dataset] = {}
        monthly_folder_paths[dataset] = None

        status = (
            "Optional dataset folder missing"
            if dataset in OPTIONAL_MONTHLY_DATASETS
            else "Required dataset folder missing"
        )

        dataset_inventory_records.append({
            "Dataset": dataset,
            "Folder": "",
            "Raster_Files_Indexed": 0,
            "Skipped_Files": 0,
            "Duplicate_Months": 0,
            "Status": status,
        })

        print(f"{dataset:<18}: ERROR - {error}")


# =====================================================================
# 6. DISCOVER STATIC RASTERS
# =====================================================================

if AUTO_DISCOVER_STATIC:

    static_rasters = discover_static_rasters(
        STATIC_FOLDER
    )

else:

    # প্রয়োজন হলে এখানে manually static raster path দেওয়া যাবে
    static_rasters = []


print()
print("=" * 78)
print("STATIC RASTERS")
print("=" * 78)

if static_rasters:

    for raster_path in static_rasters:
        print(raster_path.name)

else:
    print("No static raster will be included.")


# =====================================================================
# 7. CREATE MONTH LIST
# =====================================================================

months_to_process = get_month_list(
    START_YEAR,
    END_YEAR
)

print()
print(f"Total months to process: {len(months_to_process)}")


# =====================================================================
# 8. FIND REFERENCE RASTER
# =====================================================================

reference_path = None


for year, month in months_to_process:

    for dataset in MONTHLY_DATASETS:

        raster_path = monthly_indexes.get(
            dataset,
            {}
        ).get((year, month))

        if raster_path is not None:
            reference_path = raster_path
            break

    if reference_path is not None:
        break


if reference_path is None and static_rasters:
    reference_path = static_rasters[0]


if reference_path is None:
    raise RuntimeError(
        "কোনো reference raster পাওয়া যায়নি।"
    )


reference_metadata = get_raster_metadata(
    reference_path
)


print()
print("=" * 78)
print("REFERENCE RASTER")
print("=" * 78)

print(f"File       : {reference_path}")
print(f"CRS        : {reference_metadata['crs']}")
print(
    f"Size       : "
    f"{reference_metadata['width']} x "
    f"{reference_metadata['height']}"
)
print(f"Resolution : {reference_metadata['resolution']}")
print(f"Transform  : {reference_metadata['transform']}")
print(f"Bounds     : {reference_metadata['bounds']}")


# =====================================================================
# 9. VERIFY STATIC RASTER ALIGNMENT
# =====================================================================

static_alignment_records = []
valid_static_rasters = []


print()
print("=" * 78)
print("VERIFYING STATIC RASTERS")
print("=" * 78)


for static_path in static_rasters:

    try:

        alignment_errors = verify_raster_against_reference(
            static_path,
            reference_metadata,
            check_bounds=STRICT_BOUNDS_CHECK
        )

        if alignment_errors:

            static_alignment_records.append({
                "File": str(static_path),
                "Aligned": False,
                "Errors": " | ".join(alignment_errors),
            })

            print(f"NOT ALIGNED: {static_path.name}")

            for error in alignment_errors:
                print(f"    {error}")

        else:

            valid_static_rasters.append(static_path)

            static_alignment_records.append({
                "File": str(static_path),
                "Aligned": True,
                "Errors": "",
            })

            print(f"OK: {static_path.name}")

    except Exception as error:

        static_alignment_records.append({
            "File": str(static_path),
            "Aligned": False,
            "Errors": str(error),
        })

        print(f"FAILED TO CHECK: {static_path.name}")
        print(f"    {error}")


# =====================================================================
# 10. PROCESS EACH MONTH
# =====================================================================

processing_records = []
missing_records = []
alignment_records = []
band_inventory_records = []

successful_months = 0
failed_months = 0
skipped_existing_months = 0


print()
print("=" * 78)
print("CREATING MONTHLY PREDICTOR STACKS")
print("=" * 78)


for year, month in months_to_process:

    month_key = f"{year}_{month:02d}"

    output_path = (
        OUTPUT_FOLDER
        / f"predictor_stack_{month_key}.tif"
    )

    print()
    print("-" * 78)
    print(f"Processing: {month_key}")
    print("-" * 78)

    start_time = datetime.now()

    try:

        # -------------------------------------------------------------
        # Skip existing output
        # -------------------------------------------------------------

        if output_path.exists() and not OVERWRITE_EXISTING:

            print(f"SKIPPED: Output already exists: {output_path.name}")

            processing_records.append({
                "Year": year,
                "Month": month,
                "Month_ID": month_key,
                "Status": "Skipped existing",
                "Output_File": str(output_path),
                "Band_Count": "",
                "Processing_Seconds": 0,
                "Error": "",
            })

            skipped_existing_months += 1
            continue


        # -------------------------------------------------------------
        # Collect monthly raster paths
        # -------------------------------------------------------------

        monthly_raster_paths = []
        monthly_band_names = []
        required_missing = []

        for dataset in MONTHLY_DATASETS:

            dataset_index = monthly_indexes.get(
                dataset,
                {}
            )

            raster_path = dataset_index.get(
                (year, month)
            )

            if raster_path is None:

                missing_records.append({
                    "Year": year,
                    "Month": month,
                    "Month_ID": month_key,
                    "Dataset": dataset,
                    "Expected_Folder": str(
                        monthly_folder_paths.get(dataset)
                    ),
                    "Required": (
                        dataset
                        not in OPTIONAL_MONTHLY_DATASETS
                    ),
                })

                if dataset in OPTIONAL_MONTHLY_DATASETS:

                    print(
                        f"OPTIONAL MISSING: {dataset}"
                    )

                    continue

                required_missing.append(dataset)
                continue

            monthly_raster_paths.append(
                raster_path
            )

            monthly_band_names.append(
                dataset
            )

            print(
                f"FOUND {dataset:<18}: "
                f"{raster_path.name}"
            )


        # -------------------------------------------------------------
        # Stop if required monthly datasets are missing
        # -------------------------------------------------------------

        if required_missing:

            raise FileNotFoundError(
                "Required monthly raster missing: "
                + ", ".join(required_missing)
            )


        if not monthly_raster_paths:

            raise RuntimeError(
                f"No monthly raster found for {month_key}"
            )


        # -------------------------------------------------------------
        # Combine monthly and valid static rasters
        # -------------------------------------------------------------

        all_raster_paths = (
            monthly_raster_paths
            + valid_static_rasters
        )

        static_band_names = [
            sanitize_band_name(path.stem)
            for path in valid_static_rasters
        ]

        all_band_names = (
            monthly_band_names
            + static_band_names
        )

        all_band_names = make_unique_band_names(
            all_band_names
        )


        # -------------------------------------------------------------
        # Verify monthly raster alignment
        # -------------------------------------------------------------

        if VERIFY_ALIGNMENT:

            alignment_failed = False
            alignment_error_messages = []

            for raster_path, band_name in zip(
                all_raster_paths,
                all_band_names
            ):

                alignment_errors = (
                    verify_raster_against_reference(
                        raster_path,
                        reference_metadata,
                        check_bounds=STRICT_BOUNDS_CHECK
                    )
                )

                alignment_records.append({
                    "Year": year,
                    "Month": month,
                    "Month_ID": month_key,
                    "Band_Name": band_name,
                    "Raster_File": str(raster_path),
                    "Aligned": len(alignment_errors) == 0,
                    "Errors": " | ".join(alignment_errors),
                })

                if alignment_errors:

                    alignment_failed = True

                    message = (
                        f"{band_name}: "
                        + " | ".join(alignment_errors)
                    )

                    alignment_error_messages.append(
                        message
                    )

            if alignment_failed:

                raise ValueError(
                    "Raster alignment mismatch: "
                    + " || ".join(
                        alignment_error_messages
                    )
                )


        # -------------------------------------------------------------
        # Prepare output metadata
        # -------------------------------------------------------------

        output_profile = {
            "driver": "GTiff",
            "height": reference_metadata["height"],
            "width": reference_metadata["width"],
            "count": len(all_raster_paths),
            "dtype": OUTPUT_DTYPE,
            "crs": reference_metadata["crs"],
            "transform": reference_metadata["transform"],
            "nodata": OUTPUT_NODATA,
            "compress": COMPRESSION,
            "predictor": 3,
            "tiled": True,
            "BIGTIFF": "IF_SAFER",
        }


        # -------------------------------------------------------------
        # Write multiband predictor stack
        # -------------------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **output_profile
        ) as dst:

            for band_number, (
                raster_path,
                band_name
            ) in enumerate(
                zip(
                    all_raster_paths,
                    all_band_names
                ),
                start=1
            ):

                data = read_raster_as_float32(
                    raster_path
                )

                dst.write(
                    data,
                    band_number
                )

                dst.set_band_description(
                    band_number,
                    band_name
                )

                dst.update_tags(
                    band_number,
                    dataset=band_name,
                    source_file=str(raster_path),
                    year=year,
                    month=month,
                )

                valid_mask = (
                    data != OUTPUT_NODATA
                )

                if np.any(valid_mask):

                    valid_values = data[valid_mask]

                    minimum = float(
                        np.nanmin(valid_values)
                    )

                    maximum = float(
                        np.nanmax(valid_values)
                    )

                    mean_value = float(
                        np.nanmean(valid_values)
                    )

                    valid_pixels = int(
                        np.sum(valid_mask)
                    )

                else:

                    minimum = np.nan
                    maximum = np.nan
                    mean_value = np.nan
                    valid_pixels = 0

                band_inventory_records.append({
                    "Year": year,
                    "Month": month,
                    "Month_ID": month_key,
                    "Band_Number": band_number,
                    "Band_Name": band_name,
                    "Source_File": str(raster_path),
                    "Minimum": minimum,
                    "Maximum": maximum,
                    "Mean": mean_value,
                    "Valid_Pixels": valid_pixels,
                    "Total_Pixels": int(data.size),
                    "NoData_Pixels": int(
                        data.size - valid_pixels
                    ),
                })

            dst.update_tags(
                project="Precipitation Downscaling Khulna",
                stack_type="Monthly predictor stack",
                year=year,
                month=month,
                month_id=month_key,
                creation_time=datetime.now().isoformat(),
                band_names=",".join(all_band_names),
            )


        # -------------------------------------------------------------
        # Validate output
        # -------------------------------------------------------------

        with rasterio.open(output_path) as check:

            if check.count != len(all_raster_paths):

                raise RuntimeError(
                    f"Output band count mismatch: "
                    f"{check.count} != "
                    f"{len(all_raster_paths)}"
                )

            if check.width != reference_metadata["width"]:

                raise RuntimeError(
                    "Output width mismatch"
                )

            if check.height != reference_metadata["height"]:

                raise RuntimeError(
                    "Output height mismatch"
                )

            if check.crs != reference_metadata["crs"]:

                raise RuntimeError(
                    "Output CRS mismatch"
                )


        processing_seconds = (
            datetime.now() - start_time
        ).total_seconds()

        successful_months += 1

        processing_records.append({
            "Year": year,
            "Month": month,
            "Month_ID": month_key,
            "Status": "Success",
            "Output_File": str(output_path),
            "Band_Count": len(all_raster_paths),
            "Processing_Seconds": round(
                processing_seconds,
                3
            ),
            "Error": "",
        })

        print()
        print(f"SUCCESS: {output_path.name}")
        print(f"Bands  : {len(all_raster_paths)}")
        print(
            f"Time   : "
            f"{processing_seconds:.2f} seconds"
        )


    except Exception as error:

        failed_months += 1

        processing_seconds = (
            datetime.now() - start_time
        ).total_seconds()

        processing_records.append({
            "Year": year,
            "Month": month,
            "Month_ID": month_key,
            "Status": "Failed",
            "Output_File": str(output_path),
            "Band_Count": "",
            "Processing_Seconds": round(
                processing_seconds,
                3
            ),
            "Error": str(error),
        })

        print()
        print(f"FAILED: {month_key}")
        print(f"ERROR : {error}")


# =====================================================================
# 11. SAVE LOG FILES
# =====================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


processing_df = pd.DataFrame(
    processing_records
)

missing_df = pd.DataFrame(
    missing_records
)

alignment_df = pd.DataFrame(
    alignment_records
)

band_inventory_df = pd.DataFrame(
    band_inventory_records
)

dataset_inventory_df = pd.DataFrame(
    dataset_inventory_records
)

skipped_files_df = pd.DataFrame(
    skipped_file_records
)

duplicate_files_df = pd.DataFrame(
    duplicate_file_records
)

static_alignment_df = pd.DataFrame(
    static_alignment_records
)


processing_log_path = (
    LOG_FOLDER
    / "predictor_stack_processing_log.csv"
)

missing_log_path = (
    LOG_FOLDER
    / "predictor_stack_missing_files.csv"
)

alignment_log_path = (
    LOG_FOLDER
    / "predictor_stack_alignment_check.csv"
)

band_inventory_log_path = (
    LOG_FOLDER
    / "predictor_stack_band_inventory.csv"
)

dataset_inventory_log_path = (
    LOG_FOLDER
    / "predictor_stack_dataset_inventory.csv"
)

skipped_files_log_path = (
    LOG_FOLDER
    / "predictor_stack_unrecognized_filenames.csv"
)

duplicate_files_log_path = (
    LOG_FOLDER
    / "predictor_stack_duplicate_months.csv"
)

static_alignment_log_path = (
    LOG_FOLDER
    / "predictor_stack_static_alignment.csv"
)


processing_df.to_csv(
    processing_log_path,
    index=False
)

missing_df.to_csv(
    missing_log_path,
    index=False
)

alignment_df.to_csv(
    alignment_log_path,
    index=False
)

band_inventory_df.to_csv(
    band_inventory_log_path,
    index=False
)

dataset_inventory_df.to_csv(
    dataset_inventory_log_path,
    index=False
)

skipped_files_df.to_csv(
    skipped_files_log_path,
    index=False
)

duplicate_files_df.to_csv(
    duplicate_files_log_path,
    index=False
)

static_alignment_df.to_csv(
    static_alignment_log_path,
    index=False
)


# =====================================================================
# 12. FINAL SUMMARY
# =====================================================================

print()
print("=" * 78)
print("MONTHLY PREDICTOR STACK COMPLETED")
print("=" * 78)

print(f"Expected months         : {len(months_to_process)}")
print(f"Successful months       : {successful_months}")
print(f"Failed months           : {failed_months}")
print(f"Skipped existing months : {skipped_existing_months}")
print(f"Output folder           : {OUTPUT_FOLDER}")
print(f"Log folder              : {LOG_FOLDER}")

print()
print("Log files:")

print(f"1. {processing_log_path.name}")
print(f"2. {missing_log_path.name}")
print(f"3. {alignment_log_path.name}")
print(f"4. {band_inventory_log_path.name}")
print(f"5. {dataset_inventory_log_path.name}")
print(f"6. {skipped_files_log_path.name}")
print(f"7. {duplicate_files_log_path.name}")
print(f"8. {static_alignment_log_path.name}")


# =====================================================================
# 13. DISPLAY PROCESSING SUMMARY
# =====================================================================

if not processing_df.empty:

    print()
    print("=" * 78)
    print("PROCESSING STATUS SUMMARY")
    print("=" * 78)

    print(
        processing_df["Status"]
        .value_counts(dropna=False)
        .to_string()
    )


# =====================================================================
# 14. DISPLAY FAILED MONTHS
# =====================================================================

failed_df = processing_df[
    processing_df["Status"] == "Failed"
].copy()


if not failed_df.empty:

    print()
    print("=" * 78)
    print("FAILED MONTHS")
    print("=" * 78)

    print(
        failed_df[
            [
                "Month_ID",
                "Error"
            ]
        ].to_string(
            index=False
        )
    )


# =====================================================================
# 15. DISPLAY CREATED STACK EXAMPLE
# =====================================================================

successful_df = processing_df[
    processing_df["Status"] == "Success"
].copy()


if not successful_df.empty:

    first_output = Path(
        successful_df.iloc[0]["Output_File"]
    )

    print()
    print("=" * 78)
    print("FIRST SUCCESSFUL STACK INFORMATION")
    print("=" * 78)

    with rasterio.open(first_output) as src:

        print(f"File       : {first_output.name}")
        print(f"CRS        : {src.crs}")
        print(f"Width      : {src.width}")
        print(f"Height     : {src.height}")
        print(f"Band count : {src.count}")
        print(f"Resolution : {src.res}")
        print(f"NoData     : {src.nodata}")

        print()
        print("Band descriptions:")

        for band_number, description in enumerate(
            src.descriptions,
            start=1
        ):

            print(
                f"{band_number:02d}. "
                f"{description}"
            )

MONTHLY PREDICTOR STACK CREATION
Project root   : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna
Aligned folder : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters_aligned
Monthly folder : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters_aligned\monthly
Static folder  : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters_aligned\static
Output folder  : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\predictor_stack
Log folder     : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs

INDEXING MONTHLY RASTERS
CCS               :  72 monthly raster(s)
CDR               :  72 monthly raster(s)
CHIRPS            :  72 monthly raster(s)
ERA5              :  72 monthly raster(s)
GSMaP_Gauge       :  72 monthly raster(s)
GSMaP_MVK         :  72 monthly 